In [ ]:
# Cell 1
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.data_collection.lob_reconstructor import *
from src.features.ofi_features import compute_all_features
from src.models.kyles_lambda import estimate_kyles_lambda_ols, rolling_lambda, decompose_price_impact

plt.style.use('dark_background')
%matplotlib inline

RUN_ID = "YOUR_RUN_ID_HERE"
DATA_DIR = Path("../data")

depth  = load_depth_data(RUN_ID, DATA_DIR)
trades = load_trade_data(RUN_ID, DATA_DIR)
depth  = add_derived_columns(depth)
merged = merge_depth_and_trades(depth, trades)
df     = compute_all_features(merged)

In [ ]:
# Cell 2 — Full-sample estimate
result = estimate_kyles_lambda_ols(df['mid_return'], df['signed_volume'])
print(f"Kyle's λ = {result['lambda']:.6f}")
print(f"R²       = {result['r_squared']:.4f}")
print(f"p-value  = {result['p_value_lambda']:.4e}")
print(f"t-stat   = {result['t_stat_lambda']:.2f}")

In [ ]:
# Cell 3 — Rolling lambda plot
roll = rolling_lambda(df, window=500, step=50)

plt.figure(figsize=(13, 4))
plt.plot(roll['timestamp_ms'], roll['lambda'], color='#00bfff', lw=1.2, label='Rolling λ')
plt.axhline(result['lambda'], color='#ff6b6b', lw=1, ls='--',
            label=f"Full-sample λ = {result['lambda']:.5f}")
plt.fill_between(roll['timestamp_ms'], roll.get('ci_lower', roll['lambda']),
                 roll.get('ci_upper', roll['lambda']), alpha=0.15, color='#00bfff')
plt.ylabel("Kyle's λ"); plt.legend()
plt.title("Rolling Kyle's Lambda — does price impact change over time?")
plt.tight_layout()
plt.savefig('../data/processed/03_rolling_lambda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4 — Decomposition
decomp = decompose_price_impact(df)
print(f"\nPermanent impact: {decomp['permanent_lambda']:.6f} ({100*(1-decomp['reversion_ratio']):.1f}%)")
print(f"Temporary impact: {decomp['temporary_lambda']:.6f} ({100*decomp['reversion_ratio']:.1f}%)")